# Open PII Masking 500k – Benchmark

This notebook loads the `ai4privacy/open-pii-masking-500k-ai4privacy` dataset from Hugging Face and shows how to work with it using `pandas`.

Dataset card: `https://huggingface.co/datasets/ai4privacy/open-pii-masking-500k-ai4privacy`


In [1]:
import pandas as pd
from datasets import load_dataset

/home/gregoire/silex/privacy-service/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("ai4privacy/open-pii-masking-500k-ai4privacy")

ds

Generating validation split: 100%|██████████| 116077/116077 [00:00<00:00, 422132.07 examples/s]


DatasetDict({
    train: Dataset({
        features: ['source_text', 'masked_text', 'privacy_mask', 'split', 'uid', 'language', 'region', 'script', 'mbert_tokens', 'mbert_token_classes'],
        num_rows: 464150
    })
    validation: Dataset({
        features: ['source_text', 'masked_text', 'privacy_mask', 'split', 'uid', 'language', 'region', 'script', 'mbert_tokens', 'mbert_token_classes'],
        num_rows: 116077
    })
})

In [7]:
test_df = ds["validation"].to_pandas()
french_test_df = test_df[test_df["language"] == "fr"]
french_test_df.head()

,source_text,masked_text,privacy_mask,split,uid,language,region,script,mbert_tokens,mbert_token_classes
0,Ma mère Astrit Nani Kofi est née à Ruswil en f...,Ma mère [GIVENNAME_1] [SURNAME_1] est née à [C...,"[{'label': 'GIVENNAME', 'start': 8, 'end': 19,...",validation,5706814,fr,CH,Latn,"[Ma, mère, As, ##tri, ##t, Nan, ##i, Ko, ##fi,...","[O, O, B-GIVENNAME, I-GIVENNAME, I-GIVENNAME, ..."
7,21:00 +4026945 8718 : 'Je souhaite vous contac...,[TIME_1] [TELEPHONENUM_1] : 'Je souhaite vous ...,"[{'label': 'TIME', 'start': 0, 'end': 5, 'valu...",validation,5258182,fr,FR,Latn,"[21, :, 00, +, 402, ##69, ##45, 871, ##8, :, '...","[B-TIME, I-TIME, I-TIME, B-TELEPHONENUM, I-TEL..."
10,Lesушки Noris peuvent être atteints à rocalça@...,Lesушки [GIVENNAME_1] peuvent être atteints à ...,"[{'label': 'GIVENNAME', 'start': 8, 'end': 13,...",validation,5258368,fr,CH,Latn,"[Les, ##ушки, Nor, ##is, peuvent, être, attein...","[O, O, B-GIVENNAME, I-GIVENNAME, O, O, O, O, O..."
25,Nous remercions Dzevad Gutwirth pour son souti...,Nous remercions [GIVENNAME_1] [SURNAME_1] pour...,"[{'label': 'GIVENNAME', 'start': 16, 'end': 22...",validation,5260642,fr,CA,Latn,"[Nous, re, ##mer, ##cions, D, ##ze, ##vad, Gut...","[O, O, O, O, B-GIVENNAME, I-GIVENNAME, I-GIVEN..."
31,La poème postcoloniale est intitulée : Mtre et...,La poème postcoloniale est intitulée : [TITLE_...,"[{'label': 'TITLE', 'start': 39, 'end': 43, 'v...",validation,5261762,fr,CH,Latn,"[La, poème, post, ##colo, ##nial, ##e, est, in...","[O, O, O, O, O, O, O, O, O, B-TITLE, I-TITLE, ..."


In [13]:
from privacy_service import PrivacyService

privacy_service = PrivacyService("config.example.yaml")

test_text = french_test_df["source_text"].iloc[1]
print(test_text)

print(privacy_service.detect(test_text))

21:00 +4026945 8718 : 'Je souhaite vous contacter pour discuter de vos projets de sensibilisation au public à travers l'art du verre.'
[DetectionResult(entity_type='DATE_TIME', text='21:00', start=0, end=5, score=1.00), DetectionResult(entity_type='PHONE_NUMBER', text='+4026945 8718', start=6, end=19, score=0.40)]


In [14]:
french_test_df["detected_entities"] = french_test_df["source_text"].apply(
    lambda x: privacy_service.detect(x)
)
french_test_df.head()

,source_text,masked_text,privacy_mask,split,uid,language,region,script,mbert_tokens,mbert_token_classes,detected_entities
0,Ma mère Astrit Nani Kofi est née à Ruswil en f...,Ma mère [GIVENNAME_1] [SURNAME_1] est née à [C...,"[{'label': 'GIVENNAME', 'start': 8, 'end': 19,...",validation,5706814,fr,CH,Latn,"[Ma, mère, As, ##tri, ##t, Nan, ##i, Ko, ##fi,...","[O, O, B-GIVENNAME, I-GIVENNAME, I-GIVENNAME, ...","[DetectionResult(entity_type='DATE_TIME', text..."
7,21:00 +4026945 8718 : 'Je souhaite vous contac...,[TIME_1] [TELEPHONENUM_1] : 'Je souhaite vous ...,"[{'label': 'TIME', 'start': 0, 'end': 5, 'valu...",validation,5258182,fr,FR,Latn,"[21, :, 00, +, 402, ##69, ##45, 871, ##8, :, '...","[B-TIME, I-TIME, I-TIME, B-TELEPHONENUM, I-TEL...","[DetectionResult(entity_type='DATE_TIME', text..."
10,Lesушки Noris peuvent être atteints à rocalça@...,Lesушки [GIVENNAME_1] peuvent être atteints à ...,"[{'label': 'GIVENNAME', 'start': 8, 'end': 13,...",validation,5258368,fr,CH,Latn,"[Les, ##ушки, Nor, ##is, peuvent, être, attein...","[O, O, B-GIVENNAME, I-GIVENNAME, O, O, O, O, O...","[DetectionResult(entity_type='EMAIL_ADDRESS', ..."
25,Nous remercions Dzevad Gutwirth pour son souti...,Nous remercions [GIVENNAME_1] [SURNAME_1] pour...,"[{'label': 'GIVENNAME', 'start': 16, 'end': 22...",validation,5260642,fr,CA,Latn,"[Nous, re, ##mer, ##cions, D, ##ze, ##vad, Gut...","[O, O, O, O, B-GIVENNAME, I-GIVENNAME, I-GIVEN...","[DetectionResult(entity_type='PERSON', text='D..."
31,La poème postcoloniale est intitulée : Mtre et...,La poème postcoloniale est intitulée : [TITLE_...,"[{'label': 'TITLE', 'start': 39, 'end': 43, 'v...",validation,5261762,fr,CH,Latn,"[La, poème, post, ##colo, ##nial, ##e, est, in...","[O, O, O, O, O, O, O, O, O, B-TITLE, I-TITLE, ...","[DetectionResult(entity_type='LOCATION', text=..."


In [15]:
french_test_df[["source_text", "detected_entities", "privacy_mask"]].head()

,source_text,detected_entities,privacy_mask
0,Ma mère Astrit Nani Kofi est née à Ruswil en f...,"[DetectionResult(entity_type='DATE_TIME', text...","[{'label': 'GIVENNAME', 'start': 8, 'end': 19,..."
7,21:00 +4026945 8718 : 'Je souhaite vous contac...,"[DetectionResult(entity_type='DATE_TIME', text...","[{'label': 'TIME', 'start': 0, 'end': 5, 'valu..."
10,Lesушки Noris peuvent être atteints à rocalça@...,"[DetectionResult(entity_type='EMAIL_ADDRESS', ...","[{'label': 'GIVENNAME', 'start': 8, 'end': 13,..."
25,Nous remercions Dzevad Gutwirth pour son souti...,"[DetectionResult(entity_type='PERSON', text='D...","[{'label': 'GIVENNAME', 'start': 16, 'end': 22..."
31,La poème postcoloniale est intitulée : Mtre et...,"[DetectionResult(entity_type='LOCATION', text=...","[{'label': 'TITLE', 'start': 39, 'end': 43, 'v..."


In [35]:
from dataclasses import asdict
from IPython.display import display
import pandas as pd

# Rendre les colonnes lisibles dans les DataFrame
pd.set_option("display.max_colwidth", None)


def inspect_example(df: pd.DataFrame, idx: int) -> None:
    """Affiche proprement une ligne : texte, privacy_mask et détections.

    - df: DataFrame contenant au moins les colonnes
      `source_text`, `privacy_mask`, `detected_entities`.
    - idx: index (ou position) de la ligne à inspecter.
    """

    row = df.iloc[idx]

    print(f"Index: {idx}")
    print("\n=== Texte source ===")
    print(row["source_text"])

    print("\n=== privacy_mask (vérité terrain du dataset) ===")
    pm_df = pd.DataFrame(row["privacy_mask"])
    display(pm_df)

    print("\n=== detected_entities (PrivacyService) ===")
    det_list = [asdict(d) for d in row["detected_entities"]]
    det_df = pd.DataFrame(det_list)
    display(det_df)


# Exemple : inspecter la première ligne française
inspect_example(french_test_df, 16644)

Index: 16644

=== Texte source ===
Salut Nonso, je cherche un nouveau pot pour mon bonsaï. Pouvez-vous me recommander un magasin à Maur qui vend des pots de bonsaï ?

=== privacy_mask (vérité terrain du dataset) ===


,0
0,"{'label': 'GIVENNAME', 'start': 6, 'end': 11, 'value': 'Nonso', 'label_index': 1}"
1,"{'label': 'CITY', 'start': 96, 'end': 100, 'value': 'Maur', 'label_index': 1}"



=== detected_entities (PrivacyService) ===


,entity_type,start,end,score,text,recognizer
0,LOCATION,96,100,0.850000,Maur,SpacyRecognizer
1,LOCATION,95,100,0.566858,Maur,AI4PrivacyRecognizer
2,PERSON,0,5,0.001142,Salut,AI4PrivacyRecognizer
3,GENDER,43,47,0.000423,mon,AI4PrivacyRecognizer
4,PERSON,5,12,0.000032,"Nonso,",AI4PrivacyRecognizer
5,PERSON,85,93,0.000010,magasin,AI4PrivacyRecognizer


In [47]:
from privacy_service.recognizers.entity_mapping import map_ai4privacy_to_presidio
from typing import Dict, List, Tuple
from collections import defaultdict


def spans_overlap(det_start: int, det_end: int, gt_start: int, gt_end: int) -> bool:
    """Check if detected span overlaps with ground truth span.

    Two spans overlap if they share any character.
    """
    return not (det_end <= gt_start or det_start >= gt_end)


def spans_fully_cover(det_start: int, det_end: int, gt_start: int, gt_end: int) -> bool:
    """Check if detected span fully covers ground truth span.

    A detected entity fully covers a ground truth if:
    det_start <= gt_start and det_end >= gt_end
    """
    return det_start <= gt_start and det_end >= gt_end


def is_ground_truth_fully_covered(
    gt_start: int, gt_end: int, detected_spans: List[Tuple[int, int]]
) -> bool:
    """Check if ground truth span is fully covered by one or more detected entity spans.

    Args:
        gt_start: Ground truth start position
        gt_end: Ground truth end position
        detected_spans: List of (start, end) tuples from detected entities

    Returns:
        True if the ground truth span is fully covered by the union of detected spans
    """
    if not detected_spans:
        return False

    # Sort detected spans by start position
    sorted_spans = sorted(detected_spans, key=lambda x: x[0])

    # Check if the union of detected spans covers the ground truth
    # We need to find if there's a continuous coverage from gt_start to gt_end
    coverage_start = gt_start
    coverage_end = gt_start

    for det_start, det_end in sorted_spans:
        # If this detected span overlaps with or extends our current coverage
        if det_start <= coverage_end:
            # Extend coverage
            coverage_end = max(coverage_end, det_end)
        else:
            # Gap found, check if we've already covered the ground truth
            if coverage_start <= gt_start and coverage_end >= gt_end:
                return True
            # Reset coverage
            coverage_start = det_start
            coverage_end = det_end

    # Check final coverage
    return coverage_start <= gt_start and coverage_end >= gt_end


def evaluate_row(row: pd.Series) -> Dict:
    """Evaluate a single row by comparing detected_entities vs privacy_mask.

    New evaluation logic:
    - TP: Ground truth is fully covered by one or more detected entities (regardless of type)
    - FN: Ground truth is not detected or only partially detected
    - FP: Detected entity has no corresponding ground truth (doesn't overlap with any GT)

    Args:
        row: DataFrame row with 'detected_entities' and 'privacy_mask' columns

    Returns:
        Dictionary with metrics: tp, fp, fn, precision, recall, f1
    """
    detected = row["detected_entities"]
    ground_truth = row["privacy_mask"]

    # Prepare ground truth items (keep original info for mapping)
    gt_items = []
    for gt_item in ground_truth:
        gt_items.append(
            {
                "start": gt_item["start"],
                "end": gt_item["end"],
                "original_label": gt_item["label"],
                "mapped_type": map_ai4privacy_to_presidio(gt_item["label"]),
            }
        )

    # Track which ground truth items are fully covered
    gt_covered = [False] * len(gt_items)

    # Track which detected entities are true positives or false positives
    tp_detections = []
    fp_detections = []

    # For each detected entity, check if it overlaps with any ground truth
    for det in detected:
        overlapping_gts = []
        for i, gt in enumerate(gt_items):
            if spans_overlap(det.start, det.end, gt["start"], gt["end"]):
                overlapping_gts.append(i)

        if overlapping_gts:
            # This detection overlaps with at least one ground truth
            tp_detections.append(det)
            # Mark overlapping ground truths as potentially covered
            # (we'll check full coverage separately)
            for gt_idx in overlapping_gts:
                # Check if this detection alone fully covers the GT
                if spans_fully_cover(
                    det.start,
                    det.end,
                    gt_items[gt_idx]["start"],
                    gt_items[gt_idx]["end"],
                ):
                    gt_covered[gt_idx] = True
        else:
            # This detection doesn't overlap with any ground truth = FP
            fp_detections.append(det)

    # Check if ground truths are fully covered by the union of all detected entities
    # Collect all detected spans that overlap with each ground truth
    for i, gt in enumerate(gt_items):
        if not gt_covered[i]:
            # Collect all detected spans that overlap with this ground truth
            overlapping_spans = []
            for det in detected:
                if spans_overlap(det.start, det.end, gt["start"], gt["end"]):
                    overlapping_spans.append((det.start, det.end))

            # Check if the union of overlapping spans fully covers the ground truth
            if is_ground_truth_fully_covered(gt["start"], gt["end"], overlapping_spans):
                gt_covered[i] = True

    # Count metrics
    tp = sum(gt_covered)  # Number of ground truths fully covered
    fp = len(fp_detections)  # Detections with no corresponding ground truth
    fn = len(gt_items) - tp  # Ground truths not fully covered

    # False negatives: unmatched or partially matched ground truth items
    fn_items = [gt_items[i] for i, covered in enumerate(gt_covered) if not covered]

    # Calculate metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp_detections": tp_detections,
        "fp_detections": fp_detections,
        "fn_items": fn_items,
    }


def evaluate_dataframe(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    """Evaluate PrivacyService performance on entire DataFrame.

    Args:
        df: DataFrame with 'detected_entities' and 'privacy_mask' columns

    Returns:
        Tuple of:
        - DataFrame with per-row metrics added
        - Dictionary with aggregate metrics
    """
    print("Evaluating rows...")
    results = df.apply(evaluate_row, axis=1)

    # Add metrics columns to DataFrame
    df_eval = df.copy()
    df_eval["tp"] = [r["tp"] for r in results]
    df_eval["fp"] = [r["fp"] for r in results]
    df_eval["fn"] = [r["fn"] for r in results]
    df_eval["precision"] = [r["precision"] for r in results]
    df_eval["recall"] = [r["recall"] for r in results]
    df_eval["f1"] = [r["f1"] for r in results]

    # Aggregate metrics
    total_tp = sum(r["tp"] for r in results)
    total_fp = sum(r["fp"] for r in results)
    total_fn = sum(r["fn"] for r in results)

    aggregate_precision = (
        total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    )
    aggregate_recall = (
        total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    )
    aggregate_f1 = (
        2
        * aggregate_precision
        * aggregate_recall
        / (aggregate_precision + aggregate_recall)
        if (aggregate_precision + aggregate_recall) > 0
        else 0.0
    )

    aggregate_metrics = {
        "total_tp": total_tp,
        "total_fp": total_fp,
        "total_fn": total_fn,
        "precision": aggregate_precision,
        "recall": aggregate_recall,
        "f1": aggregate_f1,
        "num_rows": len(df),
    }

    return df_eval, aggregate_metrics


# Example: evaluate the French test DataFrame
print("Evaluating French test set...")
french_test_df_eval, metrics = evaluate_dataframe(french_test_df)

print("\n=== Aggregate Metrics ===")
print(f"Total TP: {metrics['total_tp']}")
print(f"Total FP: {metrics['total_fp']}")
print(f"Total FN: {metrics['total_fn']}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1']:.4f}")
print(f"Number of rows: {metrics['num_rows']}")

Evaluating French test set...
Evaluating rows...

=== Aggregate Metrics ===
Total TP: 45525
Total FP: 13193
Total FN: 5987
Precision: 0.7753
Recall: 0.8838
F1 Score: 0.8260
Number of rows: 22466


In [48]:
def analyze_by_entity_type(df_eval: pd.DataFrame) -> pd.DataFrame:
    """Analyze metrics broken down by entity type.

    Note: TP/FN/FP are calculated type-agnostically (based on span coverage),
    but we break down the counts by entity type for analysis.

    Args:
        df_eval: DataFrame with evaluation metrics (from evaluate_dataframe)

    Returns:
        DataFrame with per-entity-type metrics
    """
    entity_stats = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

    # Collect stats from each row
    for _, row in df_eval.iterrows():
        eval_result = evaluate_row(row)

        # Count FP by detected entity type (detections with no corresponding ground truth)
        for det in eval_result["fp_detections"]:
            entity_stats[det.entity_type]["fp"] += 1

        # Count TP/FN by ground truth entity type
        # TP: ground truths that were fully covered
        for det in eval_result["tp_detections"]:
            # Find which ground truths this detection overlaps with
            for gt_item in row["privacy_mask"]:
                if spans_overlap(det.start, det.end, gt_item["start"], gt_item["end"]):
                    mapped_type = map_ai4privacy_to_presidio(gt_item["label"])
                    # Check if this GT was fully covered (simplified: if it's in tp_detections, count it)
                    # We'll do a more accurate count below
                    pass

        # More accurate: count TP/FN by checking each ground truth
        for gt_item in row["privacy_mask"]:
            mapped_type = map_ai4privacy_to_presidio(gt_item["label"])
            gt_start = gt_item["start"]
            gt_end = gt_item["end"]

            # Check if this ground truth is fully covered
            overlapping_spans = []
            for det in row["detected_entities"]:
                if spans_overlap(det.start, det.end, gt_start, gt_end):
                    overlapping_spans.append((det.start, det.end))

            if is_ground_truth_fully_covered(gt_start, gt_end, overlapping_spans):
                entity_stats[mapped_type]["tp"] += 1
            else:
                entity_stats[mapped_type]["fn"] += 1

    # Convert to DataFrame
    entity_df = pd.DataFrame.from_dict(entity_stats, orient="index")
    entity_df = entity_df.reset_index().rename(columns={"index": "entity_type"})

    # Calculate metrics
    entity_df["precision"] = entity_df.apply(
        lambda r: r["tp"] / (r["tp"] + r["fp"]) if (r["tp"] + r["fp"]) > 0 else 0.0,
        axis=1,
    )
    entity_df["recall"] = entity_df.apply(
        lambda r: r["tp"] / (r["tp"] + r["fn"]) if (r["tp"] + r["fn"]) > 0 else 0.0,
        axis=1,
    )
    entity_df["f1"] = entity_df.apply(
        lambda r: (
            2 * r["precision"] * r["recall"] / (r["precision"] + r["recall"])
            if (r["precision"] + r["recall"]) > 0
            else 0.0
        ),
        axis=1,
    )

    # Sort by F1 score descending
    entity_df = entity_df.sort_values("f1", ascending=False)

    return entity_df


# Analyze by entity type
entity_metrics = analyze_by_entity_type(french_test_df_eval)

print("=== Metrics by Entity Type ===")
display(entity_metrics)

# Show examples of rows with high/low performance
print("\n=== Examples: High F1 Score Rows ===")
high_f1 = french_test_df_eval.nlargest(5, "f1")[["source_text", "tp", "fp", "fn", "f1"]]
display(high_f1)

print("\n=== Examples: Low F1 Score Rows (many FP/FN) ===")
low_f1 = french_test_df_eval.nsmallest(5, "f1")[["source_text", "tp", "fp", "fn", "f1"]]
display(low_f1)

=== Metrics by Entity Type ===


,entity_type,tp,fp,fn,precision,recall,f1
4,EMAIL_ADDRESS,2850,44,1,0.984796,0.999649,0.992167
2,DATE_TIME,5070,30,634,0.994118,0.888850,0.938541
13,DRIVERLICENSENUM,384,0,53,1.000000,0.878719,0.935445
6,AGE,1142,256,0,0.816881,1.000000,0.899213
8,CREDIT_CARD,392,0,102,1.000000,0.793522,0.884876
1,LOCATION,8514,2373,199,0.782034,0.977161,0.868776
7,US_SSN,441,98,40,0.818182,0.916840,0.864706
10,PASSPORTNUM,640,0,213,1.000000,0.750293,0.857334
5,IDCARDNUM,1266,0,435,1.000000,0.744268,0.853387
0,PERSON,22362,7457,1640,0.749925,0.931672,0.830977



=== Examples: High F1 Score Rows ===


,source_text,tp,fp,fn,f1
0,Ma mère Astrit Nani Kofi est née à Ruswil en février/75,4,0,0,1.0
7,21:00 +4026945 8718 : 'Je souhaite vous contacter pour discuter de vos projets de sensibilisation au public à travers l'art du verre.',2,0,0,1.0
10,Lesушки Noris peuvent être atteints à rocalça@outlook.com pour les demandes de partenariat pour les événements de danse.,2,0,0,1.0
25,Nous remercions Dzevad Gutwirth pour son soutien à notre projet de bijoux de créature fantastique. Nous sommes heureux de l'avoir comme partenaire.,2,0,0,1.0
31,La poème postcoloniale est intitulée : Mtre et est dédiée à Sazgar Gussoni.,3,0,0,1.0



=== Examples: Low F1 Score Rows (many FP/FN) ===


,source_text,tp,fp,fn,f1
165,Kijara Kessia Jatzenko a créé un nouveau groupe de discussion pour discuter de la production théâtrale surréaliste.,0,0,2,0.0
290,N'oubliez pas de joindre une copie de votre pièce d'identité et de votre justificatif de domicile à votre demande de marque déposée.,0,2,0,0.0
319,Veuillez vous assurer que vous avez votre F1907928828761 avec vous lors de votre visite.,0,1,1,0.0
343,Vérifier si l'expiration de 20-41-664-977-173 a un impact sur la valeur du produit,0,0,1,0.0
436,+541 697-325 0125: C'est très responsable de ta part ! Je peux te donner les coordonnées de [NOM DU NOTAIRE] qui pourra t'aider à créer un testament et un plan de succession.,0,0,1,0.0


In [49]:
def visualize_row_evaluation(df_eval: pd.DataFrame, idx: int) -> None:
    """Visualize TP/FP/FN for a specific row with color coding.

    Args:
        df_eval: DataFrame with evaluation metrics
        idx: Row index to visualize
    """
    row = df_eval.loc[idx]
    text = row["source_text"]

    # Get evaluation result
    eval_result = evaluate_row(row)

    print(f"Row Index: {idx}")
    print(f"\n=== Source Text ===")
    print(text)

    # Map ground truth items to their coverage status
    gt_items_with_status = []
    for gt_item in row["privacy_mask"]:
        gt_start = gt_item["start"]
        gt_end = gt_item["end"]
        mapped_type = map_ai4privacy_to_presidio(gt_item["label"])

        # Check if fully covered
        overlapping_spans = []
        for det in row["detected_entities"]:
            if spans_overlap(det.start, det.end, gt_start, gt_end):
                overlapping_spans.append((det.start, det.end))

        is_covered = is_ground_truth_fully_covered(gt_start, gt_end, overlapping_spans)
        gt_items_with_status.append(
            {
                "item": gt_item,
                "mapped_type": mapped_type,
                "is_covered": is_covered,
                "overlapping_spans": overlapping_spans,
            }
        )

    print(f"\n=== Ground Truth Status ===")
    for gt_info in gt_items_with_status:
        gt_item = gt_info["item"]
        status = (
            "✓ TP (fully covered)"
            if gt_info["is_covered"]
            else "✗ FN (not fully covered)"
        )
        print(
            f"  [{gt_item['start']:3d}-{gt_item['end']:3d}] "
            f"{gt_item['label']:15s} → {gt_info['mapped_type']:20s} {status}"
        )
        print(f"      Text: '{text[gt_item['start'] : gt_item['end']]}'")
        if gt_info["overlapping_spans"]:
            print(f"      Overlapping detections: {len(gt_info['overlapping_spans'])}")

    print(f"\n=== Detected Entities ===")
    for det in row["detected_entities"]:
        # Check if FP (doesn't overlap with any ground truth)
        is_fp = det in eval_result["fp_detections"]
        status = "✗ FP (no GT)" if is_fp else "✓ Overlaps with GT"
        print(
            f"  [{det.start:3d}-{det.end:3d}] "
            f"{det.entity_type:20s} {status:20s} "
            f"(score={det.score:.3f}) '{text[det.start : det.end]}'"
        )

    print(f"\n=== False Negatives (ground truth not fully covered) ===")
    fn_items = [
        gt_info for gt_info in gt_items_with_status if not gt_info["is_covered"]
    ]
    if fn_items:
        for gt_info in fn_items:
            gt_item = gt_info["item"]
            print(
                f"  [{gt_item['start']:3d}-{gt_item['end']:3d}] "
                f"{gt_item['label']:15s} → {gt_info['mapped_type']:20s} "
                f"'{text[gt_item['start'] : gt_item['end']]}'"
            )
            if gt_info["overlapping_spans"]:
                print(
                    f"      Partially covered by {len(gt_info['overlapping_spans'])} detection(s)"
                )
            else:
                print(f"      Not detected at all")
    else:
        print("  (none - all ground truths fully covered)")

    print(f"\n=== Row Metrics ===")
    print(f"  TP: {eval_result['tp']} (ground truths fully covered)")
    print(f"  FP: {eval_result['fp']} (detections with no corresponding ground truth)")
    print(f"  FN: {eval_result['fn']} (ground truths not fully covered)")
    print(f"  Precision: {eval_result['precision']:.4f}")
    print(f"  Recall: {eval_result['recall']:.4f}")
    print(f"  F1: {eval_result['f1']:.4f}")


# Example: visualize a specific row
print("Visualizing row with index 20506:")
visualize_row_evaluation(french_test_df_eval, 343)

Visualizing row with index 20506:
Row Index: 343

=== Source Text ===
Vérifier si l'expiration de 20-41-664-977-173 a un impact sur la valeur du produit

=== Ground Truth Status ===
  [ 28- 45] TAXNUM          → TAXNUM               ✗ FN (not fully covered)
      Text: '20-41-664-977-173'

=== Detected Entities ===

=== False Negatives (ground truth not fully covered) ===
  [ 28- 45] TAXNUM          → TAXNUM               '20-41-664-977-173'
      Not detected at all

=== Row Metrics ===
  TP: 0 (ground truths fully covered)
  FP: 0 (detections with no corresponding ground truth)
  FN: 1 (ground truths not fully covered)
  Precision: 0.0000
  Recall: 0.0000
  F1: 0.0000


In [36]:
french_test_df[["source_text", "detected_entities", "privacy_mask"]].iloc[20506]

source_text                                                                                                                                                                                                                                                                                                                                                                            Je suis vraiment satisfait de mon cahier personnalisé avec mon jadewaele@aol.com. Merci beaucoup !
detected_entities    [DetectionResult(entity_type='EMAIL_ADDRESS', text='jadewaele@aol.com', start=63, end=80, score=1.00), DetectionResult(entity_type='URL', text='aol.com', start=73, end=80, score=0.50), DetectionResult(entity_type='GENDER', text=' mon', start=58, end=62, score=0.01), DetectionResult(entity_type='EMAIL_ADDRESS', text=' jadewaele@aol.com.', start=62, end=81, score=0.00), DetectionResult(entity_type='PERSON', text=' mon', start=29, end=33, score=0.00)]
privacy_mask                                        

In [40]:
# Get metrics broken down by entity type
entity_metrics_df = analyze_by_entity_type(french_test_df_eval)

print("=== Performance Metrics by Entity Type ===")
print(f"\nTotal entity types found: {len(entity_metrics_df)}")
print("\nTop performing entity types (by F1 score):")
display(entity_metrics_df.head(10))

print("\n=== Summary Statistics ===")
print(f"Average Precision: {entity_metrics_df['precision'].mean():.4f}")
print(f"Average Recall: {entity_metrics_df['recall'].mean():.4f}")
print(f"Average F1: {entity_metrics_df['f1'].mean():.4f}")

# Show entity types with most detections
print("\n=== Entity Types by Detection Count ===")
entity_metrics_df_sorted_by_count = entity_metrics_df.sort_values(
    by="tp", ascending=False
).head(10)
display(
    entity_metrics_df_sorted_by_count[
        ["entity_type", "tp", "fp", "fn", "precision", "recall", "f1"]
    ]
)

=== Performance Metrics by Entity Type ===

Total entity types found: 19

Top performing entity types (by F1 score):


,entity_type,tp,fp,fn,precision,recall,f1
6,EMAIL_ADDRESS,5355,226,34,0.959505,0.993691,0.976299
1,PERSON,20689,17067,5245,0.547966,0.797756,0.649678
10,AGE,973,1483,169,0.396173,0.852014,0.540856
0,DATE_TIME,3400,3650,2304,0.482270,0.596073,0.533166
2,LOCATION,7503,14561,2251,0.340056,0.769223,0.471620
8,GENDER,613,5975,242,0.093048,0.716959,0.164719
11,US_SSN,313,3207,168,0.088920,0.650728,0.156461
5,TELEPHONENUM,0,0,3763,0.000000,0.000000,0.000000
4,PHONE_NUMBER,0,2175,0,0.000000,0.000000,0.000000
7,URL,0,2922,0,0.000000,0.000000,0.000000



=== Summary Statistics ===
Average Precision: 0.1530
Average Recall: 0.2830
Average F1: 0.1838

=== Entity Types by Detection Count ===


,entity_type,tp,fp,fn,precision,recall,f1
1,PERSON,20689,17067,5245,0.547966,0.797756,0.649678
2,LOCATION,7503,14561,2251,0.340056,0.769223,0.471620
6,EMAIL_ADDRESS,5355,226,34,0.959505,0.993691,0.976299
0,DATE_TIME,3400,3650,2304,0.482270,0.596073,0.533166
10,AGE,973,1483,169,0.396173,0.852014,0.540856
8,GENDER,613,5975,242,0.093048,0.716959,0.164719
11,US_SSN,313,3207,168,0.088920,0.650728,0.156461
5,TELEPHONENUM,0,0,3763,0.000000,0.000000,0.000000
4,PHONE_NUMBER,0,2175,0,0.000000,0.000000,0.000000
7,URL,0,2922,0,0.000000,0.000000,0.000000
